# 05. Interpretacja rezultatow i wnioski

**Etap z planu pracy:** interpretacja rezultatow i formulowanie wnioskow.

Notebook zbiera wyniki H1-H3 i formuluje decyzje badawcze.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205_prepared.csv"
RAW_DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
METADATA_PATH = PROJECT_ROOT / "outputs" / "prepared_dataset_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "czysta_baza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Prepared data exists:", DATA_PATH.exists())
print("Output dir:", OUTPUT_DIR)

assert DATA_PATH.exists(), f"Brakuje pliku z przygotowana baza: {DATA_PATH}"


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH: C:\Users\szymon\projekt_reddit\database\NajnowszaWersjaBazy1205_prepared.csv
Prepared data exists: True
Output dir: C:\Users\szymon\projekt_reddit\outputs\czysta_baza


In [2]:
h1 = pd.read_csv(OUTPUT_DIR / "04_h1_emotional_escalation.csv")
h2 = pd.read_csv(OUTPUT_DIR / "04_h2_cognitive_complexity.csv")
h3 = pd.read_csv(OUTPUT_DIR / "04_h3_multimodal_benchmark.csv")

display(h1)
display(h2)
display(h3)


,high_previous_anger_24h,rows,negative_links,negative_rate,anger_threshold_q75,negative_rate_delta_high_minus_other,odds_ratio_high_vs_other,fisher_p_value,chi2,chi2_p_value
0,False,49794,3845,0.0772,0.0064,-0.0046,0.9352,1.0000,0.0006,0.9802
1,True,124,9,0.0726,0.0064,-0.0046,0.9352,1.0000,0.0006,0.9802


,feature,negative_n,positive_n,negative_mean,positive_mean,difference_negative_minus_positive,negative_median,positive_median,cohens_d_negative_minus_positive,point_biserial_negative,mannwhitney_u,mannwhitney_p_value
0,Automated readability index,3854,46064,20.0715,21.5987,-1.5272,18.3295,18.5775,-0.1217,-0.0263,85139348.5000,0.0000
1,Average number of words per sentence,3854,46064,20.6283,23.0728,-2.4445,18.3333,18.4167,-0.1124,-0.0235,88506757.0000,0.7635
2,Average word length,3854,46064,5.3378,5.3320,0.0058,4.9662,4.9532,0.0044,0.0011,89823349.5000,0.2183
3,Number of words,3854,46064,276.6056,228.6261,47.9795,139.0000,104.0000,0.1288,0.0362,97562599.0000,0.0000
4,LIWC_Conj,3854,46064,0.0444,0.0405,0.0038,0.0478,0.0429,0.1386,0.0365,95900460.5000,0.0000
5,LIWC_CogMech,3854,46064,0.1248,0.1161,0.0087,0.1349,0.1250,0.1412,0.0372,96374643.5000,0.0000


,model,accuracy,macro_f1,weighted_f1,negative_precision,negative_recall,negative_f1,tn,fp,fn,tp,train_seconds
0,TF-IDF word 1-2 + Logistic Regression balanced,0.8562,0.6315,0.8800,0.2493,0.5529,0.3437,8172,1132,304,376,40.3008
1,TF-IDF + Properties + HF columns + Logistic Re...,0.8374,0.6236,0.8687,0.2350,0.6147,0.3400,7943,1361,262,418,85.8432
2,TF-IDF word 1-2 + Linear SVC balanced,0.8975,0.6320,0.9020,0.2916,0.3529,0.3194,8721,583,440,240,44.8955
3,HF sentiment columns + Properties + Logistic R...,0.7290,0.5476,0.7950,0.1603,0.7029,0.2611,6800,2504,202,478,2.3918
4,Properties + Logistic Regression balanced,0.7231,0.5428,0.7908,0.1565,0.6985,0.2557,6744,2560,205,475,2.3966
5,HF Content_Sentiment direct baseline,0.8349,0.5705,0.8616,0.1707,0.3691,0.2335,8085,1219,429,251,0.0000
6,HF sentiment columns + Logistic Regression bal...,0.7008,0.5069,0.7740,0.1209,0.5412,0.1977,6629,2675,312,368,0.0352
7,Properties + Random Forest balanced,0.9322,0.5023,0.9018,0.5600,0.0206,0.0397,9293,11,666,14,12.4704
8,Rule: high previous anger -> negative,0.9290,0.4844,0.8980,0.0606,0.0029,0.0056,9273,31,678,2,0.0000
9,Dummy most frequent,0.9319,0.4824,0.8990,0.0000,0.0000,0.0000,9304,0,680,0,0.0016


In [3]:
h1_low = h1.loc[h1["high_previous_anger_24h"] == False].iloc[0]
h1_high = h1.loc[h1["high_previous_anger_24h"] == True].iloc[0]
h1_supported = (h1_high["negative_rate"] > h1_low["negative_rate"]) and (h1_high["fisher_p_value"] < 0.05)

lower_complexity_expected = {
    "Average word length": "lower",
    "LIWC_Conj": "lower",
    "Automated readability index": "lower",
    "LIWC_CogMech": "lower",
    "Number of words": "lower",
    "Average number of words per sentence": "lower",
}

h2_eval = h2.copy()
h2_eval["supports_lower_complexity"] = h2_eval["difference_negative_minus_positive"] < 0
h2_supported_features = int(h2_eval["supports_lower_complexity"].sum())
h2_total_features = int(h2_eval.shape[0])

best_h3 = h3.sort_values(["negative_f1", "macro_f1"], ascending=False).iloc[0]
h3_supported = bool(best_h3["negative_f1"] >= 0.75)

decisions = pd.DataFrame([
    {
        "hypothesis": "H1 eskalacja emocjonalna",
        "decision": "potwierdzona" if h1_supported else "niepotwierdzona",
        "main_evidence": (
            f"negative_rate high anger={h1_high['negative_rate']:.4f}, "
            f"other={h1_low['negative_rate']:.4f}, "
            f"Fisher p={h1_high['fisher_p_value']:.4f}, "
            f"odds ratio={h1_high['odds_ratio_high_vs_other']:.3f}"
        ),
    },
    {
        "hypothesis": "H2 zlozonosc poznawcza",
        "decision": "czesciowo potwierdzona" if 0 < h2_supported_features < h2_total_features else ("potwierdzona" if h2_supported_features == h2_total_features else "niepotwierdzona"),
        "main_evidence": f"{h2_supported_features}/{h2_total_features} cech ma nizsza wartosc dla linkow negatywnych",
    },
    {
        "hypothesis": "H3 model multimodalny",
        "decision": "potwierdzona" if h3_supported else "niepotwierdzona",
        "main_evidence": f"najlepszy model: {best_h3['model']}, negative_f1={best_h3['negative_f1']:.4f}",
    },
])
display(decisions)
decisions.to_csv(OUTPUT_DIR / "05_hypothesis_decisions.csv", index=False)
h2_eval.to_csv(OUTPUT_DIR / "05_h2_direction_check.csv", index=False)


,hypothesis,decision,main_evidence
0,H1 eskalacja emocjonalna,niepotwierdzona,"negative_rate high anger=0.0726, other=0.0772,..."
1,H2 zlozonosc poznawcza,czesciowo potwierdzona,2/6 cech ma nizsza wartosc dla linkow negatywnych
2,H3 model multimodalny,niepotwierdzona,najlepszy model: TF-IDF word 1-2 + Logistic Re...


In [4]:
summary_md = f'''
## Wnioski badawcze

**H1:** {decisions.loc[0, "decision"]}. Wysoki poprzedni anger w oknie 24h nie daje istotnego wzrostu prawdopodobienstwa linku negatywnego.

**H2:** {decisions.loc[1, "decision"]}. Wyniki sugeruja roznice stylistyczne, ale nie prosty wzorzec, w ktorym wszystkie teksty negatywne sa mniej zlozone.

**H3:** {decisions.loc[2, "decision"]}. Najlepszy model nie osiaga progu F1 = 0.75 dla klasy negatywnej, mimo ze accuracy moze wygladac wysoko przy niezbalansowanej klasie.

## Ograniczenia

- Okno 24h w H1 daje mala liczbe przypadkow z historia tej samej pary.
- `LINK_SENTIMENT` i `Content_Sentiment` nie sa tym samym: pierwsza zmienna opisuje link miedzy spolecznosciami, druga sentyment tresci.
- Podzial chronologiczny jest bardziej realistyczny niz losowy, ale moze obnizac wyniki wzgledem walidacji losowej.
- Modele transformerowe wymagajace pobierania wag zostaly pozostawione jako opcjonalne rozszerzenie.

## Dalsze prace

- Przetestowac H1 dla okien 48h, 7 dni i par nieskierowanych.
- Dla H2 dodac cechy skladniowe lub embeddingowe miary zlozonosci.
- Dla H3 sprawdzic modele kosztoczulne, prog decyzyjny optymalizowany pod F1 oraz embeddingi sentence-transformers.
'''

display(Markdown(summary_md))
(OUTPUT_DIR / "05_interpretacja_wnioski.md").write_text(summary_md, encoding="utf-8")



## Wnioski badawcze

**H1:** niepotwierdzona. Wysoki poprzedni anger w oknie 24h nie daje istotnego wzrostu prawdopodobienstwa linku negatywnego.

**H2:** czesciowo potwierdzona. Wyniki sugeruja roznice stylistyczne, ale nie prosty wzorzec, w ktorym wszystkie teksty negatywne sa mniej zlozone.

**H3:** niepotwierdzona. Najlepszy model nie osiaga progu F1 = 0.75 dla klasy negatywnej, mimo ze accuracy moze wygladac wysoko przy niezbalansowanej klasie.

## Ograniczenia

- Okno 24h w H1 daje mala liczbe przypadkow z historia tej samej pary.
- `LINK_SENTIMENT` i `Content_Sentiment` nie sa tym samym: pierwsza zmienna opisuje link miedzy spolecznosciami, druga sentyment tresci.
- Podzial chronologiczny jest bardziej realistyczny niz losowy, ale moze obnizac wyniki wzgledem walidacji losowej.
- Modele transformerowe wymagajace pobierania wag zostaly pozostawione jako opcjonalne rozszerzenie.

## Dalsze prace

- Przetestowac H1 dla okien 48h, 7 dni i par nieskierowanych.
- Dla H2 dodac cechy skladniowe lub embeddingowe miary zlozonosci.
- Dla H3 sprawdzic modele kosztoczulne, prog decyzyjny optymalizowany pod F1 oraz embeddingi sentence-transformers.


1161

## Rezultat etapu

Hipotezy zostaly formalnie ocenione. Ostatni notebook przepisuje wyniki na material do raportu i prezentacji.
